# 01 Build 2.5D and Optional 3D Patch Cache

Inputs:

- raw Part 1/2/3 no-autoextract Kaggle datasets
- output dataset from notebook `00_extract_clean_manifest.ipynb`

Notebook này giải nén từng source base, tạo cache nhẹ từ raw arrays:
2.5D `uint8` memmap gồm CT multi-slice + PET projections. Training
notebooks sau đó dùng cache, không đọc raw archive lại.

Gate tối thiểu: 100% validation/test cached; train càng đủ càng tốt.

In [ ]:
from pathlib import Path
import os, re, json, shutil, subprocess, warnings
import numpy as np
import pandas as pd
from PIL import Image
import matplotlib.pyplot as plt
from IPython.display import display, Markdown
warnings.filterwarnings("ignore")

OUTPUT_ROOT = Path("/kaggle/working/vimedpet_q1_outputs") if Path("/kaggle/working").exists() else Path("vimedpet_q1_outputs")
CACHE_DIR = OUTPUT_ROOT / "01_cache"
CACHE_DIR.mkdir(parents=True, exist_ok=True)
TEMP_ROOT = Path("/kaggle/temp/vimedpet_cache_extract") if Path("/kaggle/temp").exists() else OUTPUT_ROOT / "_cache_extract"
STAGE_ROOT = Path("/kaggle/temp/vimedpet_cache_stage") if Path("/kaggle/temp").exists() else OUTPUT_ROOT / "_cache_stage"
TEMP_ROOT.mkdir(parents=True, exist_ok=True); STAGE_ROOT.mkdir(parents=True, exist_ok=True)

IMAGE_SIZE = 224
CHANNELS_25D = 6
CACHE_CASE_LIMIT = None
BUILD_3D_PATCH_CACHE = False
PATCH_DEPTH = 32
PATCH_SIZE = 128

def locate_clean_manifest():
    candidates = list(Path("/kaggle/input").rglob("q1_clean_manifest.csv")) if Path("/kaggle/input").exists() else []
    candidates += list(Path.cwd().rglob("q1_clean_manifest.csv"))
    if not candidates:
        raise FileNotFoundError("q1_clean_manifest.csv not found. Add notebook 00 output as input.")
    return sorted(candidates, key=lambda p: len(str(p)))[0]

manifest_path = locate_clean_manifest()
clean = pd.read_csv(manifest_path)
if CACHE_CASE_LIMIT:
    clean = clean.head(CACHE_CASE_LIMIT).copy()
print("Manifest:", manifest_path, clean.shape)

In [ ]:
SOURCE_BASES = sorted(clean["source_part"].dropna().unique().tolist())
INPUT_HINTS = ["/kaggle/input", "/kaggle/input/datasets", str(Path.cwd())]

def find_archive_group(source_base):
    hits = []
    for root_s in INPUT_HINTS:
        root = Path(root_s)
        if not root.exists(): continue
        for marker in list(root.rglob(f"{source_base}.zipraw")) + list(root.rglob(f"{source_base}.zip")):
            hits.append({"marker": marker, "parts": sorted(marker.parent.glob(f"{source_base}.z[0-9][0-9]"))})
    if not hits:
        raise FileNotFoundError(f"Archive marker not found for {source_base}")
    return sorted(hits, key=lambda h: len(h["parts"]), reverse=True)[0]

def stage_archive(source_base, group):
    stage = STAGE_ROOT / source_base
    if stage.exists(): shutil.rmtree(stage)
    stage.mkdir(parents=True, exist_ok=True)
    for part in group["parts"]:
        target = stage / part.name
        try: os.symlink(part, target)
        except Exception: shutil.copy2(part, target)
    zip_target = stage / f"{source_base}.zip"
    try: os.symlink(group["marker"], zip_target)
    except Exception: shutil.copy2(group["marker"], zip_target)
    return zip_target

def sevenzip_binary():
    for exe in ["7z", "7zz", "7za"]:
        try:
            subprocess.run([exe], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
            return exe
        except Exception:
            pass
    raise RuntimeError("7z/7zz/7za not found")

def extract_source(source_base):
    out = TEMP_ROOT / source_base
    if out.exists(): shutil.rmtree(out)
    out.mkdir(parents=True, exist_ok=True)
    group = find_archive_group(source_base)
    zip_path = stage_archive(source_base, group)
    cmd = [sevenzip_binary(), "x", str(zip_path), f"-o{out}", "-y"]
    print("Extract:", source_base, "parts:", len(group["parts"]))
    r = subprocess.run(cmd, text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
    print(r.stdout[-1000:])
    if r.returncode != 0: raise RuntimeError(f"7z failed: {source_base}")
    for c in [out / source_base, out / source_base / source_base, out]:
        if c.exists() and any((c / f"THANG {m}").exists() for m in range(1, 13)):
            return c
    for c in out.rglob(source_base):
        if any((c / f"THANG {m}").exists() for m in range(1, 13)):
            return c
    raise FileNotFoundError(f"Extracted root not found for {source_base}")

In [ ]:
def normalize_uint8(x, lo=None, hi=None):
    x = np.asarray(x, dtype=np.float32)
    if lo is None or hi is None:
        lo, hi = np.percentile(x, [1, 99])
    if hi <= lo:
        return np.zeros_like(x, dtype=np.uint8)
    y = np.clip((x - lo) / (hi - lo), 0, 1)
    return (y * 255).astype(np.uint8)

def resize_u8(img, size=IMAGE_SIZE):
    return np.asarray(Image.fromarray(img).resize((size, size), Image.BILINEAR), dtype=np.uint8)

def make_25d(ct, pet):
    d = ct.shape[0]
    idxs = [max(0, min(d - 1, int(d * q))) for q in [0.35, 0.50, 0.65]]
    ct_imgs = [resize_u8(normalize_uint8(ct[i], lo=-1000, hi=1000)) for i in idxs]
    pet_axial = resize_u8(normalize_uint8(np.max(pet, axis=0)))
    pet_cor = resize_u8(normalize_uint8(np.max(pet, axis=1)))
    pet_sag = resize_u8(normalize_uint8(np.max(pet, axis=2)))
    return np.stack(ct_imgs + [pet_axial, pet_cor, pet_sag], axis=0)

def make_3d_patch(vol, d=PATCH_DEPTH, size=PATCH_SIZE, lo=None, hi=None):
    z = vol.shape[0]
    if z >= d:
        start = (z - d) // 2
        patch = np.asarray(vol[start:start+d])
    else:
        pad_before = (d - z) // 2
        pad_after = d - z - pad_before
        patch = np.pad(np.asarray(vol), ((pad_before, pad_after), (0, 0), (0, 0)), mode="edge")
    frames = [resize_u8(normalize_uint8(s, lo=lo, hi=hi), size=size) for s in patch]
    return np.stack(frames, axis=0)

In [ ]:
n = len(clean)
mmap_path = CACHE_DIR / f"q1_25d_uint8_{IMAGE_SIZE}.memmap"
arr = np.memmap(mmap_path, dtype=np.uint8, mode="w+", shape=(n, CHANNELS_25D, IMAGE_SIZE, IMAGE_SIZE))
if BUILD_3D_PATCH_CACHE:
    patch_path = CACHE_DIR / f"q1_3d_uint8_d{PATCH_DEPTH}_s{PATCH_SIZE}.memmap"
    patch_arr = np.memmap(patch_path, dtype=np.uint8, mode="w+", shape=(n, 2, PATCH_DEPTH, PATCH_SIZE, PATCH_SIZE))
else:
    patch_path = None

cache_rows = []
for source_base, group in clean.groupby("source_part", sort=False):
    root = extract_source(source_base)
    for idx, row in group.iterrows():
        try:
            ct = np.load(root / row["ct_img_path"], mmap_mode="r")
            pet = np.load(root / row["pet_img_path"], mmap_mode="r")
            arr[idx] = make_25d(ct, pet)
            if BUILD_3D_PATCH_CACHE:
                patch_arr[idx, 0] = make_3d_patch(ct, lo=-1000, hi=1000)
                patch_arr[idx, 1] = make_3d_patch(pet)
            ok, err = True, ""
        except Exception as e:
            ok, err = False, str(e)
        cache_rows.append({"row_index": int(idx), "case_id": row["case_id"], "source_part": source_base, "clean_split": row["clean_split"], "cache_ok": ok, "cache_error": err})
    arr.flush()
    if BUILD_3D_PATCH_CACHE: patch_arr.flush()
    shutil.rmtree(TEMP_ROOT / source_base, ignore_errors=True)

cache_index = pd.DataFrame(cache_rows)
cache_index.to_csv(CACHE_DIR / "q1_cache_index.csv", index=False, encoding="utf-8-sig")
clean.to_csv(CACHE_DIR / "q1_clean_manifest_for_cache.csv", index=False, encoding="utf-8-sig")
meta = {
    "image_size": IMAGE_SIZE, "channels": CHANNELS_25D, "n": n,
    "mmap_path": mmap_path.name, "patch_mmap_path": patch_path.name if patch_path else None,
    "build_3d_patch_cache": BUILD_3D_PATCH_CACHE,
}
(CACHE_DIR / "q1_cache_meta.json").write_text(json.dumps(meta, indent=2), encoding="utf-8")

display(cache_index.groupby(["clean_split", "cache_ok"]).size().rename("cases").reset_index())
val_test = cache_index[cache_index.clean_split.isin(["validation", "test"])]
assert val_test["cache_ok"].all(), "Validation/test cache must be complete before final eval"

# Preview
ok_rows = cache_index[cache_index.cache_ok].head(4)
fig, axes = plt.subplots(len(ok_rows), 6, figsize=(12, 2.2 * max(1, len(ok_rows))), constrained_layout=True)
if len(ok_rows) == 1: axes = np.array([axes])
for r, (_, row) in enumerate(ok_rows.iterrows()):
    img = arr[int(row.row_index)]
    for c in range(6):
        axes[r, c].imshow(img[c], cmap="gray")
        axes[r, c].axis("off")
        axes[r, c].set_title(["CT35", "CT50", "CT65", "PET ax", "PET cor", "PET sag"][c], fontsize=8)
    axes[r, 0].set_ylabel(row.case_id, fontsize=8)
fig.savefig(CACHE_DIR / "q1_25d_cache_preview.png", dpi=160, bbox_inches="tight")
plt.show()
print("Saved cache:", CACHE_DIR)